# RT-DETRv2-R18 + P2 trên VisDrone — Kaggle (bước B–C)

Notebook chuyển từ bản chỉ `--test-only` sang train **E1 P2**. E0 là control 3 level;
E1 thêm P2 stride 4, encoder attention vẫn trên P5. Các script chuyển đổi dữ liệu,
tạo P2, export và đánh giá được nhúng sẵn; không cần gắn dataset chứa script `data111`.

**Trước khi Run All:** bật **Internet**, chọn **GPU** trong Settings và Add Input dataset
VisDrone gốc có cả train/val (`images/` và `annotations/*.txt`). Đường dẫn dataset cũ đã
được điền bên dưới; nếu Kaggle gắn đường dẫn khác, sửa `DATA_ROOT` hoặc để `None` để dò.
Notebook dùng một GPU, kể cả khi phiên có hai T4.

**Luồng chạy:** cấu hình → COCO/mapping → E0/E1 → smoke + batch thật → train → export
toàn bộ validation → APsmall/Recall-heavy + ảnh lỗi → benchmark → đóng gói output.
Mặc định `RUN_MODE="full"`, 120 epoch, batch 2, AMP, train E1. Để thử **2 epoch trước**,
đặt `RUN_MODE="proxy"`; khi train chính thức đổi về `full` và chạy lại từ đầu.
Để ablation đúng kế hoạch, chạy lại với `EXPERIMENTS=["E0_control"]` và cùng các thông số.
Không dùng P2 2 epoch để kết luận tốt hơn baseline 120 epoch.

In [ ]:
from pathlib import Path
import os, sys, json, subprocess, shutil, hashlib, math, copy

DATA_ROOT = Path("/kaggle/input/datasets/nguyendokhaihoan/aaaaaaa")  # hoặc None
RUN_MODE = "full"                         # "proxy" (2 epoch) / "full" (120 epoch)
EXPERIMENTS = ["E1_p2"]                   # hoặc ["E0_control", "E1_p2"]
BATCH_SIZE = 2                            # cùng batch cho E0/E1; giảm về 1 nếu OOM
VAL_BATCH_SIZE = 2
NUM_WORKERS = 2
SEED = 42
USE_AMP = True
RUN_SMOKE = True
RUN_REAL_PREFLIGHT = True
RUN_BENCHMARK = True

# Chỉ resume last.pth của ĐÚNG experiment và mode đã chạy bằng notebook này.
# Gắn output phiên trước vào Input; thư mục checkpoint phải có run_config.yml,
# run_manifest.json, log.txt và best.pth. Để None khi train mới.
RESUME = {"E0_control": None, "E1_p2": None}
# Ví dụ: RESUME["E1_p2"] = "/kaggle/input/my-p2-output/full/E1_p2/last.pth"

# Bước B cho baseline cũ là tùy chọn. Không nạp B2 vào E1 để train.
RUN_B2_EVALUATION = False
B2_CHECKPOINT = Path("/kaggle/input/datasets/nguyendokhaihoan/133333333/best.pth")
# Nếu có YAML thực tế lúc train baseline, gắn vào Input và điền tại đây.
# Nếu None: dùng kiến trúc R18 / 3 decoder layer / 11 đầu ra của notebook cũ.
B2_CONFIG = None

REPO_COMMIT = "29320b6fd828f8e0987a71426cf2d961b09dfed7"
WORK = Path("/kaggle/working")
REPO = WORK / "RT-DETR" / "rtdetrv2_pytorch"
SCRIPTS = WORK / "rtdetrv2_next"
COCO_ROOT = WORK / "visdrone_coco"
OUT = WORK / "rtdetrv2_experiments" / RUN_MODE
CONFIGS = OUT / "configs"
DEVICE = "cuda:0"
assert RUN_MODE in {"proxy", "full"}
assert EXPERIMENTS and len(EXPERIMENTS) == len(set(EXPERIMENTS))
assert set(EXPERIMENTS) <= {"E0_control", "E1_p2"}
assert BATCH_SIZE >= 1 and VAL_BATCH_SIZE >= 1 and NUM_WORKERS >= 0
for path in (SCRIPTS, OUT, CONFIGS):
    path.mkdir(parents=True, exist_ok=True)

def run(args, cwd=None, log=None):
    args = [str(x) for x in args]
    print(" ".join(args), flush=True)
    stream = Path(log).open("a", encoding="utf-8") if log else None
    try:
        with subprocess.Popen(args, cwd=cwd, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
            for line in process.stdout:
                print(line, end="", flush=True)
                if stream:
                    stream.write(line)
            status = process.wait()
        if status:
            raise subprocess.CalledProcessError(status, args)
    finally:
        if stream:
            stream.close()


## 1. Repo và môi trường

Giữ PyTorch/CUDA sẵn có của Kaggle; chỉ cài thư viện cần cho train/evaluate.
Repo được khóa commit để E0/E1 dùng cùng mã nguồn. Sửa nhỏ AMP để `cuda:0` dùng đúng
`device.type`, và giữ best AP của các epoch trước khi resume. Không thay evaluator trong vòng train.

In [ ]:
run([sys.executable, "-m", "pip", "install", "-q", "faster-coco-eval>=1.6.6,<2",
     "pycocotools>=2.0.7", "PyYAML>=6", "scipy", "tensorboard", "packaging", "Pillow"])
import torch, torchvision, yaml
if not torch.cuda.is_available():
    raise RuntimeError("Chọn GPU trong Kaggle Settings rồi khởi động lại phiên.")
print("torch:", torch.__version__, "torchvision:", torchvision.__version__)
print("GPU:", torch.cuda.get_device_name(0), "VRAM GB:",
      round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
if not REPO.exists():
    run(["git", "clone", "https://github.com/lyuwenyu/RT-DETR.git", REPO.parent])
actual_commit = subprocess.check_output(
    ["git", "-C", str(REPO.parent), "rev-parse", "HEAD"], text=True).strip()
if actual_commit != REPO_COMMIT:
    run(["git", "-C", REPO.parent, "checkout", REPO_COMMIT])

engine = REPO / "src/solver/det_engine.py"
text = engine.read_text(encoding="utf-8")
text = text.replace("device_type=str(device)", "device_type=device.type")
engine.write_text(text, encoding="utf-8")

# Official solver initializes best_stat from scratch, even on resume.
solver = REPO / "src/solver/det_solver.py"
text = solver.read_text(encoding="utf-8")
needle = "        best_stat = {'epoch': -1, }"
replacement = """        best_stat = {'epoch': -1, }
        # Kaggle resume: retain the best AP from previous completed validations.
        if self.cfg.resume and self.output_dir:
            history = self.output_dir / 'log.txt'
            if history.exists():
                for line in history.read_text().splitlines():
                    row = json.loads(line)
                    values = row.get('test_coco_eval_bbox')
                    if (values and row['epoch'] <= self.last_epoch
                            and values[0] > best_stat.get('coco_eval_bbox', -1)):
                        best_stat = {'epoch': row['epoch'], 'coco_eval_bbox': values[0]}
"""
if "# Kaggle resume:" not in text:
    if text.count(needle) != 1:
        raise RuntimeError("Solver khác phiên bản đã kiểm tra; kiểm tra REPO_COMMIT.")
    solver.write_text(text.replace(needle, replacement), encoding="utf-8")


## 2. Script nhúng sẵn

Các cell `%%writefile` chỉ tạo công cụ trong `/kaggle/working`; chạy nguyên cell.

In [ ]:
%%writefile /kaggle/working/rtdetrv2_next/prepare_visdrone_kaggle.py
#!/usr/bin/env python3
"""
Prepare VisDrone2019-DET for RT-DETR training on Kaggle.

Supported input layouts
-----------------------

Official extracted layout:

  <root>/
    VisDrone2019-DET-train/
      images/*.jpg
      annotations/*.txt
    VisDrone2019-DET-val/
      images/*.jpg
      annotations/*.txt

Kaggle layout with an extra VisDrone2019 directory:

  <root>/
    VisDrone2019/
      VisDrone2019-DET-train/
        images/
        annotations/
      VisDrone2019-DET-val/
        images/
        annotations/

Kaggle layout with duplicated split directories:

  <root>/
    VisDrone2019/
      VisDrone2019-DET-train/
        VisDrone2019-DET-train/
          images/
          annotations/
      VisDrone2019-DET-val/
        VisDrone2019-DET-val/
          images/
          annotations/

The script also performs a recursive fallback search for the split folders.

VisDrone annotation format
--------------------------

Each row is:

  bbox_left,bbox_top,bbox_width,bbox_height,score,
  object_category,truncation,occlusion

Classes 1..10 are retained.

Category 0 (ignored regions) and category 11 (others) are excluded because
they are not part of the standard 10-class VisDrone detection label set used
by most RT-DETR recipes.

Output
------

The output directory contains:

  instances_train.json
  instances_val.json
  bad_images_train.json
  bad_images_val.json
  dataset_stats.json
  classes.txt
  rtdetr_dataset.json
"""

from __future__ import annotations

import argparse
import json
import sys
from collections import Counter, defaultdict
from pathlib import Path

try:
    from PIL import Image
except ImportError:
    print("Pillow is required. Run: pip install -q pillow", file=sys.stderr)
    raise


CLASSES = [
    "pedestrian",
    "people",
    "bicycle",
    "car",
    "van",
    "truck",
    "tricycle",
    "awning-tricycle",
    "bus",
    "motor",
]

CLASS_ID = {i + 1: name for i, name in enumerate(CLASSES)}

VALID_IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp"}


def is_valid_split_dir(path: Path) -> bool:
    """Return True when a directory contains both images/ and annotations/."""
    return (
        path.is_dir()
        and (path / "images").is_dir()
        and (path / "annotations").is_dir()
    )


def find_split(root: Path, split: str) -> Path:
    """
    Find a VisDrone split directory.

    This supports both the normal VisDrone layout and the nested Kaggle
    structure visible in datasets such as:

      VisDrone2019/
        VisDrone2019-DET-train/
          VisDrone2019-DET-train/
            images/
            annotations/
    """
    root = root.expanduser().resolve()

    split_name = f"VisDrone2019-DET-{split}"

    # Most common layouts first.
    candidates = [
        # Official layout
        root / split_name,

        # Some archives may preserve capitalization differently.
        root / f"VisDrone2019-DET-{split.title()}",

        # Simple train/val directory.
        root / split,

        # Kaggle dataset containing an outer VisDrone2019 directory.
        root / "VisDrone2019" / split_name,

        # Your current Kaggle structure:
        # VisDrone2019/VisDrone2019-DET-train/VisDrone2019-DET-train/
        root / "VisDrone2019" / split_name / split_name,

        # Duplicated split folder without outer VisDrone2019.
        root / split_name / split_name,

        # If --input already points at VisDrone2019.
        root / split_name / split_name,
    ]

    # Remove duplicate paths while preserving order.
    unique_candidates = []
    seen = set()

    for candidate in candidates:
        key = str(candidate)
        if key not in seen:
            unique_candidates.append(candidate)
            seen.add(key)

    for candidate in unique_candidates:
        if is_valid_split_dir(candidate):
            print(f"[OK] Found {split}: {candidate}")
            return candidate

    # Recursive fallback.
    #
    # This makes the script resilient to extra archive/Kaggle folder levels.
    recursive_matches = []

    if root.exists():
        try:
            for candidate in root.rglob(split_name):
                if is_valid_split_dir(candidate):
                    recursive_matches.append(candidate)

                # Handle duplicated folder:
                # .../VisDrone2019-DET-train/VisDrone2019-DET-train/
                nested = candidate / split_name
                if is_valid_split_dir(nested):
                    recursive_matches.append(nested)
        except PermissionError:
            pass

    # De-duplicate fallback matches.
    recursive_matches = sorted(
        {p.resolve() for p in recursive_matches},
        key=lambda p: (len(p.parts), str(p)),
    )

    if recursive_matches:
        selected = recursive_matches[0]
        print(f"[OK] Found {split} recursively: {selected}")

        if len(recursive_matches) > 1:
            print(
                f"[WARN] Multiple '{split}' directories found. "
                f"Using the first one: {selected}"
            )
            for other in recursive_matches[1:]:
                print(f"       alternative: {other}")

        return selected

    tried = "\n".join(f"  - {p}" for p in unique_candidates)

    raise FileNotFoundError(
        f"\nCannot find VisDrone '{split}' split under:\n"
        f"  {root}\n\n"
        f"The script requires a directory containing BOTH:\n"
        f"  images/\n"
        f"  annotations/\n\n"
        f"Paths checked:\n{tried}\n"
    )


def read_rows(path: Path):
    """Read one VisDrone annotation text file."""
    if not path.exists():
        return []

    rows = []

    for line_no, line in enumerate(
        path.read_text(
            encoding="utf-8",
            errors="replace",
        ).splitlines(),
        1,
    ):
        line = line.strip()

        if not line:
            continue

        parts = [x.strip() for x in line.split(",")]

        if len(parts) < 8:
            print(
                f"WARN {path}:{line_no}: "
                f"expected at least 8 comma-separated fields, got {len(parts)}"
            )
            continue

        try:
            x, y, w, h = map(float, parts[:4])
            score = int(float(parts[4]))
            cls = int(float(parts[5]))
            trunc = int(float(parts[6]))
            occ = int(float(parts[7]))
        except ValueError:
            print(f"WARN {path}:{line_no}: malformed numeric row")
            continue

        rows.append((x, y, w, h, score, cls, trunc, occ))

    return rows


def clamp_box(x, y, w, h, width, height):
    """Clamp a bounding box to the image boundaries."""
    x1 = max(0.0, min(float(width), x))
    y1 = max(0.0, min(float(height), y))

    x2 = max(0.0, min(float(width), x + w))
    y2 = max(0.0, min(float(height), y + h))

    return (
        x1,
        y1,
        max(0.0, x2 - x1),
        max(0.0, y2 - y1),
    )


def process_split(
    split_dir: Path,
    split: str,
    out_dir: Path,
    stats: dict,
):
    """Convert one VisDrone split to COCO detection JSON."""
    images_dir = split_dir / "images"
    ann_dir = split_dir / "annotations"

    print()
    print("=" * 80)
    print(f"Processing split : {split}")
    print(f"Split directory  : {split_dir}")
    print(f"Images directory : {images_dir}")
    print(f"Labels directory : {ann_dir}")
    print("=" * 80)

    coco = {
        "info": {
            "description": f"VisDrone2019-DET-{split} prepared for RT-DETR"
        },
        "licenses": [],
        "images": [],
        "annotations": [],
        "categories": [
            {
                "id": i,
                "name": name,
                "supercategory": "vehicle/person",
            }
            for i, name in CLASS_ID.items()
        ],
    }

    image_files = sorted(
        p
        for p in images_dir.iterdir()
        if p.is_file() and p.suffix.lower() in VALID_IMAGE_SUFFIXES
    )

    if not image_files:
        raise RuntimeError(
            f"No images found in:\n  {images_dir}"
        )

    print(f"Found {len(image_files)} image files.")

    ann_id = 1
    bad_images = []
    missing_annotations = 0

    # COCO image IDs must correspond only to successfully loaded images.
    next_image_id = 1

    for index, image_path in enumerate(image_files, 1):
        if index == 1 or index % 500 == 0 or index == len(image_files):
            print(
                f"[{split}] image {index}/{len(image_files)} "
                f"({index / len(image_files) * 100:.1f}%)"
            )

        try:
            with Image.open(image_path) as im:
                width, height = im.size
                im.verify()
        except Exception as exc:
            bad_images.append(
                {
                    "file": str(image_path),
                    "error": repr(exc),
                }
            )
            continue

        image_id = next_image_id
        next_image_id += 1

        coco["images"].append(
            {
                "id": image_id,
                "file_name": image_path.name,
                "width": width,
                "height": height,
            }
        )

        annotation_path = ann_dir / f"{image_path.stem}.txt"

        if not annotation_path.exists():
            missing_annotations += 1

        rows = read_rows(annotation_path)

        for x, y, w, h, score, cls, trunc, occ in rows:
            stats["raw_rows"] += 1
            stats["raw_classes"][str(cls)] += 1

            # VisDrone:
            # class 0  = ignored region
            # class 11 = others
            # score 0  = ignored sample
            if (
                score == 0
                or cls not in CLASS_ID
                or w <= 0
                or h <= 0
            ):
                stats["ignored_rows"] += 1
                continue

            x, y, w, h = clamp_box(
                x,
                y,
                w,
                h,
                width,
                height,
            )

            area = w * h

            if area <= 0:
                stats["ignored_rows"] += 1
                continue

            if area < 32 * 32:
                size = "small"
            elif area < 96 * 96:
                size = "medium"
            else:
                size = "large"

            occ_name = {
                0: "none",
                1: "partial",
                2: "heavy",
            }.get(occ, "unknown")

            coco["annotations"].append(
                {
                    "id": ann_id,
                    "image_id": image_id,
                    "category_id": cls,
                    "bbox": [
                        round(x, 3),
                        round(y, 3),
                        round(w, 3),
                        round(h, 3),
                    ],
                    "area": round(area, 3),
                    "iscrowd": 0,
                    "segmentation": [],
                    "ignore": 0,
                    "truncation": trunc,
                    "occlusion": occ,
                }
            )

            ann_id += 1

            stats["kept_rows"] += 1
            stats["classes"][CLASS_ID[cls]] += 1
            stats["sizes"][size] += 1
            stats["occlusion"][occ_name] += 1
            stats["per_split"][split]["annotations"] += 1

    out_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    instances_path = out_dir / f"instances_{split}.json"
    bad_images_path = out_dir / f"bad_images_{split}.json"

    instances_path.write_text(
        json.dumps(
            coco,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    bad_images_path.write_text(
        json.dumps(
            bad_images,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    stats["per_split"][split]["images"] = len(coco["images"])
    stats["per_split"][split]["bad_images"] = len(bad_images)
    stats["per_split"][split]["missing_annotations"] = missing_annotations

    print(
        f"[DONE] {split}: "
        f"{len(coco['images'])} images, "
        f"{len(coco['annotations'])} annotations, "
        f"{len(bad_images)} bad images, "
        f"{missing_annotations} missing annotation files"
    )

    print(f"COCO JSON: {instances_path}")

    return coco


def main():
    parser = argparse.ArgumentParser(
        description="Prepare VisDrone2019-DET as COCO JSON for RT-DETR."
    )

    parser.add_argument(
        "--input",
        required=True,
        help=(
            "Dataset root. The script automatically detects official and "
            "nested Kaggle VisDrone directory layouts."
        ),
    )

    parser.add_argument(
        "--output",
        required=True,
        help=(
            "Output directory, for example "
            "/kaggle/working/visdrone_coco"
        ),
    )

    args = parser.parse_args()

    root = Path(args.input).expanduser().resolve()
    out = Path(args.output).expanduser().resolve()

    print(f"Input root : {root}")
    print(f"Output dir : {out}")

    if not root.exists():
        raise FileNotFoundError(
            f"Input root does not exist:\n  {root}"
        )

    # Resolve once so we do not repeatedly search the entire dataset.
    train_dir = find_split(root, "train")
    val_dir = find_split(root, "val")

    stats = {
        "classes": Counter(),
        "raw_classes": Counter(),
        "sizes": Counter(),
        "occlusion": Counter(),
        "raw_rows": 0,
        "kept_rows": 0,
        "ignored_rows": 0,
        "per_split": defaultdict(
            lambda: {
                "images": 0,
                "annotations": 0,
                "bad_images": 0,
                "missing_annotations": 0,
            }
        ),
    }

    process_split(
        train_dir,
        "train",
        out,
        stats,
    )

    process_split(
        val_dir,
        "val",
        out,
        stats,
    )

    # Convert defaultdict / Counter objects into plain JSON-compatible dicts.
    stats = json.loads(
        json.dumps(
            stats,
            default=dict,
        )
    )

    out.mkdir(
        parents=True,
        exist_ok=True,
    )

    (out / "dataset_stats.json").write_text(
        json.dumps(
            stats,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    (out / "classes.txt").write_text(
        "\n".join(CLASSES) + "\n",
        encoding="utf-8",
    )

    config = {
        "dataset_root": str(out),
        "train_annotations": str(out / "instances_train.json"),
        "val_annotations": str(out / "instances_val.json"),
        "train_images": str((train_dir / "images").resolve()),
        "val_images": str((val_dir / "images").resolve()),
        "num_classes": len(CLASSES),
        "class_names": CLASSES,
    }

    (out / "rtdetr_dataset.json").write_text(
        json.dumps(
            config,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    print()
    print("=" * 80)
    print("DATASET PREPARATION COMPLETE")
    print("=" * 80)
    print(json.dumps(stats, indent=2, ensure_ascii=False))
    print()
    print(f"Prepared COCO files in: {out}")
    print(f"Train images: {train_dir / 'images'}")
    print(f"Val images  : {val_dir / 'images'}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/rtdetrv2_next/export_predictions.py
"""Export validation predictions using the official RT-DETRv2 loader."""
from __future__ import annotations

import argparse
import hashlib
import json
import os
from pathlib import Path
import subprocess
import sys

import torch


def import_config(repo):
    repo = Path(repo).resolve()
    if not (repo / 'src' / 'core').is_dir():
        raise ValueError('--repo must point to rtdetrv2_pytorch (contains src/core).')
    sys.path.insert(0, str(repo))
    from src.core import YAMLConfig
    return YAMLConfig


def checkpoint_weights(checkpoint, use_model=False):
    if not use_model and checkpoint.get('ema') is not None:
        return checkpoint['ema']['module'], 'ema'
    return checkpoint['model'], 'model'


def label_mapping(categories, mode, map_file=None):
    if map_file:
        data = json.loads(Path(map_file).read_text(encoding='utf-8'))
        mapping = {int(key): int(value) if value is not None else None
                   for key, value in data.items()}
    elif mode == 'category':
        mapping = {int(c['id']): int(c['id']) for c in categories}
    else:
        mapping = {index: int(c['id']) for index, c in enumerate(categories)}
    ids = {c['id'] for c in categories}
    retained = [value for value in mapping.values() if value is not None]
    if set(retained) != ids or len(retained) != len(ids):
        raise ValueError('Label mapping must map bijectively onto GT category IDs.')
    return mapping


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--repo', required=True, type=Path)
    parser.add_argument('--config', required=True, type=Path)
    parser.add_argument('--checkpoint', required=True, type=Path)
    parser.add_argument('--annotations', required=True, type=Path)
    parser.add_argument('--output', required=True, type=Path)
    labels = parser.add_mutually_exclusive_group(required=True)
    labels.add_argument('--label-mode', choices=['category', 'contiguous0'])
    labels.add_argument('--label-map', type=Path)
    parser.add_argument('--device', default='cuda:0')
    parser.add_argument('--workers', type=int, default=0)
    parser.add_argument('--batch-size', type=int, default=1)
    parser.add_argument('--model-weights', action='store_true', help='Use model instead of EMA.')
    args = parser.parse_args()
    if args.batch_size < 1 or args.workers < 0:
        parser.error('batch-size must be positive and workers nonnegative.')
    args.repo = args.repo.resolve()
    args.config = args.config.resolve()
    args.checkpoint = args.checkpoint.resolve()
    args.annotations = args.annotations.resolve()
    args.output = args.output.resolve()
    YAMLConfig = import_config(args.repo)
    # Match repo-relative dataset paths and any local modifications in the config.
    os.chdir(args.repo)
    cfg = YAMLConfig(str(args.config))
    cfg.yaml_cfg.setdefault('PResNet', {})['pretrained'] = False
    cfg.yaml_cfg['val_dataloader'].pop('total_batch_size', None)
    cfg.yaml_cfg['val_dataloader'].update(batch_size=args.batch_size,
                                         num_workers=args.workers,
                                         shuffle=False, drop_last=False)
    if args.device.startswith('cuda') and not torch.cuda.is_available():
        raise RuntimeError('CUDA unavailable; use the training environment or --device cpu.')
    annotations = json.loads(args.annotations.read_text(encoding='utf-8'))
    mapping = label_mapping(annotations['categories'], args.label_mode, args.label_map)
    checkpoint = torch.load(args.checkpoint, map_location='cpu', weights_only=True)
    state, source = checkpoint_weights(checkpoint, args.model_weights)
    model = cfg.model
    model.load_state_dict(state, strict=True)
    del state, checkpoint
    device = torch.device(args.device)
    model.to(device).eval()
    postprocessor = cfg.postprocessor.to(device).eval()
    loader = cfg.val_dataloader
    expected = {row['id']: row['file_name'] for row in annotations['images']}
    actual = {row['id']: row['file_name'] for row in loader.dataset.coco.dataset['images']}
    if actual != expected:
        raise ValueError('Config validation image IDs/names differ from --annotations.')
    predictions, seen = [], []
    with torch.inference_mode():
        for step, (images, targets) in enumerate(loader, 1):
            sizes = torch.stack([target['orig_size'] for target in targets]).to(device)
            outputs = postprocessor(model(images.to(device)), sizes)
            for target, output in zip(targets, outputs):
                image_id = int(target['image_id'].item())
                seen.append(image_id)
                for box, score, label in zip(output['boxes'].cpu().tolist(),
                                             output['scores'].cpu().tolist(),
                                             output['labels'].cpu().tolist()):
                    if label not in mapping:
                        raise ValueError(f'Unmapped output label {label}; check training class mapping.')
                    if mapping[label] is None:
                        continue
                    x1, y1, x2, y2 = box
                    if x2 <= x1 or y2 <= y1:
                        continue
                    predictions.append({'image_id': image_id, 'category_id': mapping[label],
                                        'bbox': [x1, y1, x2-x1, y2-y1], 'score': score})
            if step % 50 == 0:
                print(f'Validation batches: {step}/{len(loader)}')
    if len(seen) != len(set(seen)) or set(seen) != set(expected):
        raise ValueError('Loader did not cover validation exactly once.')
    args.output.mkdir(parents=True, exist_ok=True)
    (args.output / 'predictions.json').write_text(json.dumps(predictions, allow_nan=False),
                                                encoding='utf-8')
    try:
        commit = subprocess.check_output(['git', '-C', str(args.repo), 'rev-parse', 'HEAD'],
                                          text=True, stderr=subprocess.DEVNULL).strip()
    except (OSError, subprocess.CalledProcessError):
        commit = None
    metadata = {'image_ids': seen,
                'annotations_sha256': hashlib.sha256(args.annotations.read_bytes()).hexdigest(),
                'export': {'config': str(args.config), 'checkpoint': str(args.checkpoint),
                           'weights': source, 'label_map': mapping, 'repo_commit': commit,
                           'torch': torch.__version__, 'device': str(device),
                           'precision': 'float32', 'score_filter': None}}
    (args.output / 'coverage.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
    import yaml
    (args.output / 'resolved_config.yml').write_text(
        yaml.safe_dump(cfg.yaml_cfg, sort_keys=False), encoding='utf-8')
    print(f'Exported {len(predictions)} detections from {len(seen)} images to {args.output}')


if __name__ == '__main__':
    main()


In [ ]:
%%writefile /kaggle/working/rtdetrv2_next/prepare_p2.py
"""Generate paired E0/E1 configurations from an existing RT-DETRv2 baseline."""
from __future__ import annotations

import argparse
import copy
import os
from pathlib import Path

import torch
import yaml

from export_predictions import import_config


def add_p2(base):
    result = copy.deepcopy(base)
    architecture = result['RTDETR']
    if (architecture['backbone'] != 'PResNet' or architecture['encoder'] != 'HybridEncoder'
            or architecture['decoder'] != 'RTDETRTransformerv2'):
        raise ValueError('P2 preparation supports official PResNet/HybridEncoder/RTDETRTransformerv2 only.')
    backbone, encoder, decoder = (result[key] for key in
                                  ['PResNet', 'HybridEncoder', 'RTDETRTransformerv2'])
    depth = backbone['depth']
    if depth not in [18, 34, 50, 101]:
        raise ValueError(f'Unsupported PResNet depth: {depth}')
    if (backbone.get('return_idx', [1, 2, 3]) != [1, 2, 3]
            or encoder.get('feat_strides', [8, 16, 32]) != [8, 16, 32]
            or decoder.get('num_levels', 3) != 3):
        raise ValueError('Expected an unmodified three-level P3/P4/P5 baseline.')
    channels = [64, 128, 256, 512] if depth < 50 else [256, 512, 1024, 2048]
    hidden = encoder.get('hidden_dim', 256)
    if encoder.get('in_channels') != channels[1:]:
        raise ValueError('Baseline encoder channels do not match PResNet depth.')
    if decoder.get('hidden_dim', 256) != hidden:
        raise ValueError('Encoder and decoder hidden_dim must agree.')
    points = decoder.get('num_points', 4)
    if isinstance(points, list):
        if len(points) != 3:
            raise ValueError('Expected three decoder sampling-point counts.')
        points = [points[0], *points]
    backbone['return_idx'] = [0, 1, 2, 3]
    encoder.update(in_channels=channels, feat_strides=[4, 8, 16, 32],
                   use_encoder_idx=[index+1 for index in encoder.get('use_encoder_idx', [2])])
    decoder.update(feat_channels=[hidden]*4, feat_strides=[4, 8, 16, 32],
                   num_levels=4, num_points=points)
    return result


def smoke(YAMLConfig, path, device):
    cfg = YAMLConfig(str(path))
    cfg.yaml_cfg['PResNet']['pretrained'] = False
    model = cfg.model.to(device)
    size = cfg.yaml_cfg.get('eval_spatial_size', [640, 640])
    if not size or any(s % 32 for s in size):
        raise ValueError('Smoke test needs eval_spatial_size divisible by 32.')
    images = torch.rand(2, 3, *size, device=device)
    model.eval()
    with torch.no_grad():
        output = model(images)
    assert output['pred_boxes'].shape[-1] == 4
    assert torch.isfinite(output['pred_logits']).all()
    model.train()
    targets = [{'labels': torch.tensor([0], device=device),
                'boxes': torch.tensor([[0.5, 0.5, 0.1, 0.1]], device=device)} for _ in range(2)]
    loss = sum(cfg.criterion.to(device)(model(images, targets), targets).values())
    if not torch.isfinite(loss):
        raise RuntimeError('Non-finite smoke-test loss.')
    loss.backward()
    if any(p.grad is not None and not torch.isfinite(p.grad).all() for p in model.parameters()):
        raise RuntimeError('Non-finite smoke-test gradient.')
    print(f'Smoke passed: {path.name}; loss={loss.item():.4f}')


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--repo', required=True, type=Path)
    parser.add_argument('--base-config', required=True, type=Path)
    parser.add_argument('--output', required=True, type=Path)
    parser.add_argument('--smoke', action='store_true')
    parser.add_argument('--device', default='cpu')
    args = parser.parse_args()
    args.repo, args.base_config, args.output = (p.resolve() for p in
                                               [args.repo, args.base_config, args.output])
    YAMLConfig = import_config(args.repo)
    os.chdir(args.repo)
    base = YAMLConfig(str(args.base_config)).yaml_cfg
    base.pop('__include__', None)
    p2 = add_p2(base)
    args.output.mkdir(parents=True, exist_ok=True)
    paths = []
    for name, config in [('E0_control', base), ('E1_p2', p2)]:
        config['output_dir'] = str(args.output / name)
        path = args.output / f'{name}.yml'
        path.write_text(yaml.safe_dump(config, sort_keys=False), encoding='utf-8')
        paths.append(path)
        print(f'Config: {path}')
    if args.smoke:
        for path in paths:
            torch.manual_seed(42)
            smoke(YAMLConfig, path, torch.device(args.device))


if __name__ == '__main__':
    main()


In [ ]:
%%writefile /kaggle/working/rtdetrv2_next/evaluate.py
"""COCO bbox metrics and matched-GT small/occlusion diagnostics."""
from __future__ import annotations

import argparse
from collections import Counter, defaultdict
import copy
import csv
import hashlib
import json
import math
from pathlib import Path

import numpy as np
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

VEHICLES = {'bicycle', 'car', 'van', 'truck', 'tricycle',
            'awning-tricycle', 'bus', 'motor'}
METRICS = ['AP', 'AP50', 'AP75', 'APsmall', 'APmedium', 'APlarge',
           'AR1', 'AR10', 'AR100', 'ARsmall', 'ARmedium', 'ARlarge']


def write_json(path, data):
    Path(path).write_text(json.dumps(data, indent=2, ensure_ascii=False,
                                   allow_nan=False), encoding='utf-8')


def write_csv(path, rows, fields):
    with Path(path).open('w', encoding='utf-8-sig', newline='') as stream:
        writer = csv.DictWriter(stream, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)


def load_results(gt, predictions):
    if predictions:
        return gt.loadRes(copy.deepcopy(predictions))
    result = COCO()
    result.dataset = {'images': copy.deepcopy(gt.dataset['images']),
                      'categories': copy.deepcopy(gt.dataset['categories']),
                      'annotations': []}
    result.createIndex()
    return result


def validate_predictions(gt, predictions, coverage):
    expected = set(gt.getImgIds())
    seen = coverage['image_ids']
    if len(seen) != len(set(seen)) or set(seen) != expected:
        raise ValueError('Coverage must include every validation image exactly once.')
    categories = set(gt.getCatIds())
    for row in predictions:
        if row['image_id'] not in expected or row['category_id'] not in categories:
            raise ValueError(f'Unknown image/category ID: {row}')
        box = row['bbox']
        if (len(box) != 4 or not all(math.isfinite(x) for x in box)
                or box[2] <= 0 or box[3] <= 0
                or not math.isfinite(row['score']) or not 0 <= row['score'] <= 1):
            raise ValueError(f'Invalid xywh box/score: {row}')


def coco_metrics(gt, predictions, cat_ids=None):
    evaluator = COCOeval(gt, load_results(gt, predictions), 'bbox')
    evaluator.params.imgIds = sorted(gt.getImgIds())
    if cat_ids is not None:
        evaluator.params.catIds = sorted(cat_ids)
    evaluator.evaluate()
    evaluator.accumulate()
    evaluator.summarize()
    metrics = {key: float(value) if value >= 0 else None
               for key, value in zip(METRICS, evaluator.stats)}
    per_class = []
    for index, category_id in enumerate(evaluator.params.catIds):
        row = {'category_id': category_id, 'class': gt.cats[category_id]['name']}
        for key, area in [('AP', 0), ('APsmall', 1)]:
            values = evaluator.eval['precision'][:, :, index, area, -1]
            valid = values[values >= 0]
            row[key] = float(valid.mean()) if valid.size else None
        per_class.append(row)
    return metrics, per_class


def box_iou(box, other):
    x, y, w, h = box
    xx, yy, ww, hh = other
    intersection = max(0., min(x+w, xx+ww)-max(x, xx)) * max(
        0., min(y+h, yy+hh)-max(y, yy))
    union = w*h + ww*hh - intersection
    return intersection / union if union > 0 else 0.


def group_names(annotation):
    area = annotation['area']
    size = 'small' if area <= 1024 else 'medium' if area <= 9216 else 'large'
    occlusion = {0: 'none', 1: 'partial', 2: 'heavy'}.get(
        annotation.get('occlusion'), 'unknown')
    return ['all', size, occlusion, f'{size}_{occlusion}']


def diagnostics(gt, predictions, score=0.3, iou=0.5, vehicle_ids=None):
    selected = [row for row in predictions if row['score'] >= score]
    dt = load_results(gt, selected)
    ev = COCOeval(gt, dt, 'bbox')
    ev.params.iouThrs = np.array([iou])
    ev.params.areaRng = [[0, 1e10]]
    ev.params.areaRngLbl = ['all']
    ev.params.maxDets = [100]
    ev.evaluate()
    matched, evaluated_dt, ignored_dt = set(), set(), set()
    for result in ev.evalImgs:
        if result is None:
            continue
        matched.update(gid for gid, did, ignore in zip(
            result['gtIds'], result['gtMatches'][0], result['gtIgnore'])
                       if did > 0 and not ignore)
        for did, ignore in zip(result['dtIds'], result['dtIgnore'][0]):
            evaluated_dt.add(did)
            if ignore:
                ignored_dt.add(did)
    matched_dt = {int(did) for result in ev.evalImgs if result is not None
                  for did in result['gtMatches'][0] if did > 0}
    by_image = defaultdict(list)
    gt_by_image = defaultdict(list)
    for did in evaluated_dt - ignored_dt:
        by_image[dt.anns[did]['image_id']].append(dt.anns[did])
    for annotation in gt.anns.values():
        if not annotation.get('iscrowd', 0):
            gt_by_image[annotation['image_id']].append(annotation)

    counts = defaultdict(Counter)
    errors, image_counts = [], defaultdict(Counter)
    for annotations in gt_by_image.values():
        for annotation in annotations:
            detected = annotation['id'] in matched
            scopes = ['all_classes']
            if annotation['category_id'] in vehicle_ids:
                scopes.append('vehicles')
            for scope in scopes:
                for group in group_names(annotation):
                    counts[(scope, group)]['gt'] += 1
                    counts[(scope, group)]['tp'] += int(detected)
            if annotation['category_id'] not in vehicle_ids:
                continue
            image_id = annotation['image_id']
            image_counts[image_id]['gt_vehicles'] += 1
            if detected:
                image_counts[image_id]['tp_vehicles'] += 1
                continue
            candidates = by_image[image_id]
            same = [box_iou(annotation['bbox'], p['bbox']) for p in candidates
                    if p['category_id'] == annotation['category_id']]
            wrong = [box_iou(annotation['bbox'], p['bbox']) for p in candidates
                     if p['category_id'] != annotation['category_id']]
            reason = ('class_confusion' if max(wrong, default=0) >= iou else
                      'localization' if max(same, default=0) >= 0.1 else
                      'miss')
            image_counts[image_id]['fn_vehicles'] += 1
            groups = group_names(annotation)
            if 'small' in groups:
                image_counts[image_id]['fn_small'] += 1
            if 'heavy' in groups:
                image_counts[image_id]['fn_heavy'] += 1
            errors.append({'image_id': image_id, 'kind': 'FN', 'reason': reason,
                           'category_id': annotation['category_id'],
                           'gt_id': annotation['id'], 'bbox': annotation['bbox'],
                           'groups': groups})
    for did in sorted(evaluated_dt - matched_dt - ignored_dt):
        prediction = dt.anns[did]
        if prediction['category_id'] not in vehicle_ids:
            continue
        annotations = gt_by_image[prediction['image_id']]
        same = [box_iou(prediction['bbox'], a['bbox']) for a in annotations
                if a['category_id'] == prediction['category_id']]
        wrong = [box_iou(prediction['bbox'], a['bbox']) for a in annotations
                 if a['category_id'] != prediction['category_id']]
        reason = ('duplicate' if max(same, default=0) >= iou else
                  'class_confusion' if max(wrong, default=0) >= iou else
                  'localization' if max(same, default=0) >= 0.1 else 'background')
        image_counts[prediction['image_id']]['fp_vehicles'] += 1
        errors.append({'image_id': prediction['image_id'], 'kind': 'FP',
                       'reason': reason, 'category_id': prediction['category_id'],
                       'prediction_id': did, 'bbox': prediction['bbox'],
                       'score': prediction['score']})
    groups = []
    for scope in ['all_classes', 'vehicles']:
        for group in ['all', 'small', 'medium', 'large', 'none', 'partial',
                      'heavy', 'unknown', 'small_none', 'small_partial',
                      'small_heavy', 'small_unknown']:
            count = counts[(scope, group)]
            n, tp = count['gt'], count['tp']
            groups.append({'scope': scope, 'group': group, 'gt': n, 'tp': tp,
                           'fn': n-tp, 'recall': tp/n if n else None})
    rows = []
    for image_id in sorted(gt.getImgIds()):
        count = image_counts[image_id]
        rows.append({'image_id': image_id,
                     'file_name': gt.imgs[image_id]['file_name'],
                     **{key: count[key] for key in ['gt_vehicles', 'tp_vehicles',
                        'fn_vehicles', 'fn_small', 'fn_heavy', 'fp_vehicles']}})
    rows.sort(key=lambda r: (-r['fn_small']-r['fn_heavy'], -r['fn_vehicles'],
                             r['image_id']))
    return groups, errors, rows


def render_failures(gt, errors, rows, image_root, output, limit):
    from PIL import Image, ImageDraw
    grouped = defaultdict(list)
    for error in errors:
        grouped[error['image_id']].append(error)
    output.mkdir(exist_ok=True)
    for row in [r for r in rows if grouped[r['image_id']]][:limit]:
        with Image.open(image_root / row['file_name']) as source:
            image = source.convert('RGB')
        draw = ImageDraw.Draw(image)
        for error in grouped[row['image_id']]:
            x, y, w, h = error['bbox']
            color = 'red' if error['kind'] == 'FN' else 'yellow'
            draw.rectangle([x, y, x+w, y+h], outline=color, width=2)
            label = f"{error['kind']} {gt.cats[error['category_id']]['name']}"
            draw.text((max(0, x), max(0, y-12)), label, fill=color,
                      stroke_width=1, stroke_fill='black')
        image.save(output / f"{row['image_id']:06d}.jpg")


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--annotations', required=True, type=Path)
    parser.add_argument('--predictions', required=True, type=Path)
    parser.add_argument('--coverage', required=True, type=Path)
    parser.add_argument('--output', required=True, type=Path)
    parser.add_argument('--score', type=float, default=0.3)
    parser.add_argument('--iou', type=float, default=0.5)
    parser.add_argument('--images', type=Path)
    parser.add_argument('--visualize', type=int, default=20)
    args = parser.parse_args()
    if not 0 <= args.score <= 1 or not 0.1 < args.iou <= 1:
        parser.error('score must be in [0,1], IoU in (0.1,1].')
    gt = COCO(str(args.annotations))
    predictions = json.loads(args.predictions.read_text(encoding='utf-8'))
    coverage = json.loads(args.coverage.read_text(encoding='utf-8'))
    digest = hashlib.sha256(args.annotations.read_bytes()).hexdigest()
    if coverage.get('annotations_sha256') != digest:
        raise ValueError('Coverage manifest does not correspond to this GT file.')
    validate_predictions(gt, predictions, coverage)
    vehicle_ids = {c['id'] for c in gt.dataset['categories'] if c['name'] in VEHICLES}
    if not vehicle_ids:
        raise ValueError('No VisDrone vehicle category names found in GT.')
    overall, per_class = coco_metrics(gt, predictions)
    vehicles, _ = coco_metrics(gt, predictions, vehicle_ids)
    groups, errors, rows = diagnostics(gt, predictions, args.score, args.iou, vehicle_ids)
    args.output.mkdir(parents=True, exist_ok=True)
    summary = {
        'protocol': {'evaluation': 'COCO bbox; converted VisDrone, not official challenge',
                     'maxDets': [1, 10, 100], 'diagnostic_score': args.score,
                     'diagnostic_iou': args.iou,
                     'diagnostic_maxDets': '100 per image per category (COCO)',
                     'diagnostic_small': 'GT area <= 1024 original-image pixels',
                     'recall_definition': 'micro TP/GT after matching all GT; not COCO AR',
                     'annotations_sha256': digest,
                     'coverage_images': len(coverage['image_ids']),
                     'vehicle_categories': sorted(vehicle_ids),
                     'export': coverage.get('export', {})},
        'all_classes': overall, 'vehicles': vehicles, 'groups': groups,
        'error_counts': dict(Counter(f"{e['kind']}_{e['reason']}" for e in errors)),
        'warnings': ['Ignored regions were excluded by the original converter; '
                     'these results are not official VisDrone challenge metrics.']}
    write_json(args.output / 'metrics.json', summary)
    write_json(args.output / 'errors.json', errors)
    write_csv(args.output / 'recall_groups.csv', groups,
              ['scope', 'group', 'gt', 'tp', 'fn', 'recall'])
    write_csv(args.output / 'per_class.csv', per_class,
              ['category_id', 'class', 'AP', 'APsmall'])
    write_csv(args.output / 'failure_images.csv', rows,
              ['image_id', 'file_name', 'gt_vehicles', 'tp_vehicles', 'fn_vehicles',
               'fn_small', 'fn_heavy', 'fp_vehicles'])
    if args.images and args.visualize > 0:
        render_failures(gt, errors, rows, args.images, args.output / 'failures',
                        args.visualize)
    print(f'Results: {args.output.resolve()}')


if __name__ == '__main__':
    main()


## 3. VisDrone → COCO và kiểm tra mapping

Giữ category **1–10**, `remap_mscoco_category=False`, `num_classes=11` như notebook cũ.
Classifier có slot 0 dự phòng; exporter bỏ slot này bằng JSON mapping tường minh.
Đây là classifier 11 đầu ra cho **10 class VisDrone**, không thêm class `others`.
GT train/val giữ nguyên occlusion/truncation để đo bước B. Converter bỏ ignored regions;
đánh giá là COCO trên VisDrone đã chuyển đổi, không phải challenge VisDrone chính thức.

In [ ]:
if DATA_ROOT is None or not DATA_ROOT.exists():
    DATA_ROOT = Path("/kaggle/input")
run([sys.executable, SCRIPTS / "prepare_visdrone_kaggle.py",
     "--input", DATA_ROOT, "--output", COCO_ROOT])
dataset = json.loads((COCO_ROOT / "rtdetr_dataset.json").read_text())
stats = json.loads((COCO_ROOT / "dataset_stats.json").read_text())
TRAIN_IMAGES = Path(dataset["train_images"])
VAL_IMAGES = Path(dataset["val_images"])
TRAIN_JSON = COCO_ROOT / "instances_train.json"
VAL_JSON = COCO_ROOT / "instances_val.json"
NUM_CLASSES = 11
LABEL_MAP = COCO_ROOT / "label_map.json"
LABEL_MAP.write_text(json.dumps({i: i if i else None for i in range(11)}, indent=2))
for split, ann_path, image_root in [("train", TRAIN_JSON, TRAIN_IMAGES),
                                     ("val", VAL_JSON, VAL_IMAGES)]:
    ann = json.loads(ann_path.read_text())
    ids = {c["id"] for c in ann["categories"]}
    assert ids == set(range(1, 11))
    assert ann["images"] and ann["annotations"]
    assert all(1 <= a["category_id"] < NUM_CLASSES for a in ann["annotations"])
    assert all("occlusion" in a for a in ann["annotations"])
    missing = [i["file_name"] for i in ann["images"]
               if not (image_root / i["file_name"]).is_file()]
    if missing or stats["per_split"][split]["missing_annotations"]:
        raise RuntimeError(f"{split}: thiếu ảnh/annotation; kiểm tra dataset. {missing[:5]}")
    print(split, "images:", len(ann["images"]), "annotations:", len(ann["annotations"]),
          "bad images:", stats["per_split"][split]["bad_images"])
del ann
GT_HASH = hashlib.sha256(VAL_JSON.read_bytes()).hexdigest()
TRAIN_HASH = hashlib.sha256(TRAIN_JSON.read_bytes()).hexdigest()


## 4. Cấu hình E0/E1 (bước C)

Resolve recipe R18 chính thức rồi thay đường dẫn dữ liệu, batch và mapping cho Kaggle.
Giữ input **640×640**, 300 queries, 3 decoder layers, AdamW LR 1e-4 và augmentation của recipe.
Full: 120 epoch, augmentation dừng epoch 117, schedule gốc. Proxy: 2 epoch, LR giảm ở
epoch 1, augmentation dừng epoch 1, warmup rút ngắn. E0/E1 có cùng ngân sách trong mỗi mode.
Checkpoint phụ mỗi 20 epoch; luôn có `last.pth` và `best.pth` để hạn chế dung lượng output.
Batch Kaggle có thể khác lần train B2 cũ: dùng E0 mới để kết luận đóng góp của P2.

In [ ]:
sys.path.insert(0, str(REPO))
from src.core import YAMLConfig
base = copy.deepcopy(YAMLConfig(str(REPO / "configs/rtdetrv2/rtdetrv2_r18vd_120e_coco.yml")).yaml_cfg)
base.pop("__include__", None)
base.update(num_classes=NUM_CLASSES, remap_mscoco_category=False,
            use_amp=USE_AMP, sync_bn=False, print_freq=100, checkpoint_freq=20)
for split, ann_file, images, batch in [
    ("train", TRAIN_JSON, TRAIN_IMAGES, BATCH_SIZE),
    ("val", VAL_JSON, VAL_IMAGES, VAL_BATCH_SIZE),
]:
    loader = base[f"{split}_dataloader"]
    loader.pop("total_batch_size", None)
    loader.update(batch_size=batch, num_workers=NUM_WORKERS,
                  shuffle=(split == "train"), drop_last=(split == "train"))
    loader["dataset"].update(img_folder=str(images), ann_file=str(ann_file))
if RUN_MODE == "proxy":
    base["epoches"] = 2
    base["lr_scheduler"].update(milestones=[1], gamma=0.1)
    base["lr_warmup_scheduler"]["warmup_duration"] = 100
    base["ema"]["warmups"] = 100
    base["train_dataloader"]["dataset"]["transforms"]["policy"]["epoch"] = 1
    base["train_dataloader"]["collate_fn"]["stop_epoch"] = 1
else:
    base["train_dataloader"]["collate_fn"]["stop_epoch"] = 117
BASE_CONFIG = CONFIGS / "visdrone_base.yml"
BASE_CONFIG.write_text(yaml.safe_dump(base, sort_keys=False), encoding="utf-8")
run([sys.executable, SCRIPTS / "prepare_p2.py", "--repo", REPO,
     "--base-config", BASE_CONFIG, "--output", CONFIGS], cwd=REPO)
CONFIG_PATHS = {name: CONFIGS / f"{name}.yml" for name in ["E0_control", "E1_p2"]}
for name, path in CONFIG_PATHS.items():
    cfg = yaml.safe_load(path.read_text())
    cfg["output_dir"] = str(OUT / name)
    path.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding="utf-8")
    print(name, "strides:", cfg["RTDETRTransformerv2"]["feat_strides"],
          "encoder attention index:", cfg["HybridEncoder"]["use_encoder_idx"],
          "epochs:", cfg["epoches"], "batch:", BATCH_SIZE)


In [ ]:
%%writefile /kaggle/working/rtdetrv2_next/preflight.py

import argparse, sys, torch
from pathlib import Path
from export_predictions import import_config
from prepare_p2 import smoke

p = argparse.ArgumentParser()
p.add_argument('--repo', required=True)
p.add_argument('--config', required=True)
p.add_argument('--device', default='cuda:0')
p.add_argument('--synthetic', action='store_true')
a = p.parse_args()
torch.set_num_threads(4)
torch.manual_seed(42)
C = import_config(a.repo)
device = torch.device(a.device)
if a.synthetic:
    smoke(C, Path(a.config), device)
else:
    cfg = C(a.config)
    cfg.yaml_cfg['PResNet']['pretrained'] = False
    for split in ['train', 'val']:
        cfg.yaml_cfg[f'{split}_dataloader']['num_workers'] = 0
    model = cfg.model.to(device).train()
    criterion = cfg.criterion.to(device)
    optimizer = cfg.optimizer
    loader = cfg.train_dataloader
    loader.set_epoch(0)
    samples, targets = next(iter(loader))
    assert samples.shape[1:] == (3, 640, 640)
    for t in targets:
        assert t['boxes'].shape[-1] == 4 and torch.isfinite(t['boxes']).all()
        assert not t['labels'].numel() or (t['labels'].min() >= 1 and t['labels'].max() <= 10)
    samples = samples.to(device)
    targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
    amp = cfg.yaml_cfg.get('use_amp', False) and device.type == 'cuda'
    scaler = torch.amp.GradScaler('cuda', enabled=amp)
    with torch.autocast(device_type=device.type, enabled=amp):
        outputs = model(samples, targets=targets)
    with torch.autocast(device_type=device.type, enabled=False):
        loss = sum(criterion(outputs, targets, epoch=0, step=0, global_step=0).values())
    assert torch.isfinite(loss), loss
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    assert all(torch.isfinite(p.grad).all() for p in model.parameters() if p.grad is not None)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 0.1)
    scaler.step(optimizer)
    scaler.update()
    optimizer.zero_grad(set_to_none=True)
    model.eval()
    images, targets = next(iter(cfg.val_dataloader))
    with torch.inference_mode():
        sizes = torch.stack([t['orig_size'] for t in targets]).to(device)
        result = cfg.postprocessor.to(device)(model(images.to(device)), sizes)
    assert len(result) == len(targets)
    assert all(torch.isfinite(r['boxes']).all() for r in result)
    print(f'Real-data preflight passed: {Path(a.config).name}; loss={loss.item():.4f}')


## 5. Kiểm tra trước train

Smoke kiểm tra inference + train forward/backward của cả E0 và E1. Batch thật kiểm tra
augmentation, nhãn, loss/gradient, một bước optimizer với AMP và validation postprocessor
cho experiment được chọn. Chạy trong subprocess để trả lại VRAM trước khi train.
Preflight không tải pretrained; vòng train sẽ tải cùng backbone pretrained cho E0/E1.

In [ ]:
if RUN_SMOKE:
    for name, path in CONFIG_PATHS.items():
        run([sys.executable, SCRIPTS / "preflight.py", "--repo", REPO,
             "--config", path, "--device", DEVICE, "--synthetic"], cwd=REPO)
if RUN_REAL_PREFLIGHT:
    for name in EXPERIMENTS:
        run([sys.executable, SCRIPTS / "preflight.py", "--repo", REPO,
             "--config", CONFIG_PATHS[name], "--device", DEVICE], cwd=REPO)


## 6. Train E1 P2 (hoặc E0 control)

Train mới dùng backbone pretrained, không dùng `best.pth` 3 level cũ để resume/tuning P2.
Để tiếp tục phiên Kaggle sau: lưu output hiện tại, gắn output đó vào Input và đặt `RESUME`
trỏ tới `last.pth`, giữ nguyên mode/batch/seed/config. Notebook kiểm tra manifest, ngân sách,
mapping và checkpoint rồi chép log/best vào output mới. Checkpoint lưu ở cuối epoch;
phần epoch đang chạy dở sẽ chạy lại khi resume.
Không đổi batch âm thầm nếu OOM: giảm `BATCH_SIZE`, chạy lại cấu hình/preflight và train mới;
E0 dùng cùng batch mới. Chạy từng experiment ở từng phiên nếu không đủ thời gian cho cả hai.

In [ ]:
def resume_signature(cfg):
    cfg = copy.deepcopy(cfg)
    cfg.pop("output_dir", None)
    cfg.pop("__include__", None)
    for split in ("train", "val"):
        ds = cfg[f"{split}_dataloader"]["dataset"]
        ds.pop("img_folder", None)
        ds.pop("ann_file", None)
    return cfg

def manifest_for(name, cfg):
    return {"experiment": name, "mode": RUN_MODE, "seed": SEED,
            "repo_commit": REPO_COMMIT, "train_sha256": TRAIN_HASH,
            "val_sha256": GT_HASH, "label_map": json.loads(LABEL_MAP.read_text()),
            "signature": resume_signature(cfg)}

for name in EXPERIMENTS:
    path = CONFIG_PATHS[name]
    cfg = yaml.safe_load(path.read_text())
    folder = OUT / name
    folder.mkdir(parents=True, exist_ok=True)
    current_manifest = manifest_for(name, cfg)
    resume_path = Path(RESUME[name]) if RESUME[name] else None
    if resume_path:
        previous = resume_path.parent
        for required in (resume_path.name, "run_config.yml", "run_manifest.json", "log.txt", "best.pth"):
            if not (previous / required).is_file():
                raise FileNotFoundError(f"Resume cần {previous / required}")
        old_manifest = json.loads((previous / "run_manifest.json").read_text())
        if old_manifest != current_manifest:
            raise ValueError("Resume khác architecture/budget/data/seed. Chỉ resume đúng experiment và mode.")
        old_cfg = yaml.safe_load((previous / "run_config.yml").read_text())
        assert resume_signature(old_cfg) == resume_signature(cfg)
        checkpoint = torch.load(resume_path, map_location="cpu", weights_only=True)
        state = checkpoint["model"]
        levels = len({k.split(".")[2] for k in state if k.startswith("decoder.input_proj.")})
        if levels != cfg["RTDETRTransformerv2"]["num_levels"]:
            raise ValueError(f"Checkpoint có {levels} level; không khớp experiment {name}.")
        assert state["decoder.enc_score_head.weight"].shape[0] == NUM_CLASSES
        assert all(k in checkpoint for k in ["optimizer", "lr_scheduler", "last_epoch"])
        epoch = checkpoint["last_epoch"]
        history = [json.loads(line) for line in (previous / "log.txt").read_text().splitlines()]
        if not history or max(row["epoch"] for row in history) > epoch:
            raise ValueError("Dùng last.pth tương ứng log.txt; checkpoint cũ hơn log sẽ mất nhất quán best.")
        if epoch >= cfg["epoches"] - 1:
            raise ValueError("Checkpoint đã hoàn tất ngân sách này; không cần resume.")
        del state, checkpoint
        if previous.resolve() != folder.resolve():
            if (folder / "log.txt").exists() or (folder / "last.pth").exists():
                raise FileExistsError(f"Output {folder} đã có lượt train; dùng thư mục/phiên mới.")
            for filename in ("log.txt", "best.pth"):
                shutil.copy2(previous / filename, folder / filename)
    elif (folder / "last.pth").exists() or (folder / "log.txt").exists():
        raise FileExistsError(f"{folder} đã có train. Đặt RESUME trỏ tới last.pth để tiếp tục.")
    shutil.copy2(path, folder / "run_config.yml")
    (folder / "run_manifest.json").write_text(json.dumps(current_manifest, indent=2), encoding="utf-8")
    command = [sys.executable, "-u", REPO / "tools/train.py", "-c", path,
               "--seed", SEED, "--device", DEVICE, "--output-dir", folder]
    if USE_AMP:
        command.append("--use-amp")
    if resume_path:
        command.extend(["--resume", resume_path])
    run(command, cwd=REPO, log=folder / "console.log")
    assert (folder / "best.pth").exists()


## 7. Export + small/occlusion (bước B)

Nạp **EMA** từ best checkpoint bằng `strict=True`, đúng validation transforms và cùng GT.
Không lọc confidence trước AP. Recall nhóm dùng confidence 0.3, IoU 0.5, matching trên toàn
bộ GT trước khi chia nhóm; đây không phải COCO ARsmall. Xuất `metrics.json`, `per_class.csv`,
`recall_groups.csv`, `failure_images.csv`, `errors.json` và 20 ảnh lỗi.

`RUN_B2_EVALUATION=True` thêm kết quả B2 từ checkpoint cũ. Nếu không có config thực tế,
config R18/11 đầu ra là giả định lấy từ notebook cũ: phải xác nhận protocol trước khi so AP.

In [ ]:
def export_evaluate(name, config, checkpoint):
    dest = OUT / name / "export"
    run([sys.executable, SCRIPTS / "export_predictions.py", "--repo", REPO,
         "--config", config, "--checkpoint", checkpoint, "--annotations", VAL_JSON,
         "--label-map", LABEL_MAP, "--device", DEVICE, "--workers", NUM_WORKERS,
         "--batch-size", VAL_BATCH_SIZE, "--output", dest], cwd=REPO)
    run([sys.executable, SCRIPTS / "evaluate.py", "--annotations", VAL_JSON,
         "--predictions", dest / "predictions.json", "--coverage", dest / "coverage.json",
         "--score", "0.3", "--iou", "0.5", "--images", VAL_IMAGES,
         "--output", dest / "evaluation"])

for name in EXPERIMENTS:
    export_evaluate(name, CONFIG_PATHS[name], OUT / name / "best.pth")
if RUN_B2_EVALUATION:
    if not B2_CHECKPOINT.is_file():
        raise FileNotFoundError(B2_CHECKPOINT)
    b2 = copy.deepcopy(YAMLConfig(str(B2_CONFIG or BASE_CONFIG)).yaml_cfg)
    b2.pop("__include__", None)
    b2["val_dataloader"]["dataset"].update(img_folder=str(VAL_IMAGES), ann_file=str(VAL_JSON))
    b2_path = CONFIGS / "B2_eval.yml"
    b2_path.write_text(yaml.safe_dump(b2, sort_keys=False), encoding="utf-8")
    export_evaluate("B2", b2_path, B2_CHECKPOINT)


In [ ]:
%%writefile /kaggle/working/rtdetrv2_next/benchmark.py

import argparse, json, time
from pathlib import Path
import numpy as np
import torch
from export_predictions import import_config, checkpoint_weights

p = argparse.ArgumentParser()
p.add_argument('--repo', required=True)
p.add_argument('--config', required=True)
p.add_argument('--checkpoint', required=True)
p.add_argument('--output', required=True)
p.add_argument('--device', default='cuda:0')
a = p.parse_args()
cfg = import_config(a.repo)(a.config)
cfg.yaml_cfg['PResNet']['pretrained'] = False
model = cfg.model
checkpoint = torch.load(a.checkpoint, map_location='cpu', weights_only=True)
state, source = checkpoint_weights(checkpoint)
model.load_state_dict(state, strict=True)
del state, checkpoint
device = torch.device(a.device)
model = model.to(device).eval()
x = torch.rand(1, 3, *cfg.yaml_cfg['eval_spatial_size'], device=device)
def sync():
    if device.type == 'cuda':
        torch.cuda.synchronize(device)
with torch.inference_mode():
    for _ in range(50):
        model(x)
    sync()
    times = []
    for _ in range(200):
        sync()
        start = time.perf_counter()
        model(x)
        sync()
        times.append(1000 * (time.perf_counter() - start))
mean = float(np.mean(times))
result = {'Latency_ms': mean, 'Latency_median_ms': float(np.median(times)),
          'Latency_std_ms': float(np.std(times)), 'FPS': 1000 / mean,
          'Parameters_M': sum(p.numel() for p in model.parameters()) / 1e6,
          'protocol': {'gpu': torch.cuda.get_device_name(device) if device.type == 'cuda' else 'cpu',
                       'device': str(device), 'torch': str(torch.__version__),
                       'input': list(x.shape), 'precision': 'float32', 'backend': 'PyTorch eager',
                       'warmup': 50, 'runs': 200, 'weights': source,
                       'scope': 'model forward only, excludes preprocessing/postprocessing/I/O'}}
Path(a.output).write_text(json.dumps(result, indent=2))
print(json.dumps(result, indent=2))


## 8. Benchmark riêng sau train

Batch 1, 640×640, FP32, PyTorch eager, warmup 50 lần, đo 200 lần và CUDA synchronize.
Đây là latency model, không gồm đọc ảnh/resize/postprocessor/vẽ/ghi video.
Train dùng AMP nhưng benchmark E0/E1 cùng FP32. Chỉ đối chiếu FPS trên cùng GPU/protocol;
mốc 85% phải dùng baseline đo lại, không lấy trực tiếp FPS cũ khác GPU.

In [ ]:
if RUN_BENCHMARK:
    jobs = [(name, CONFIG_PATHS[name], OUT / name / "best.pth") for name in EXPERIMENTS]
    if RUN_B2_EVALUATION:
        jobs.append(("B2", b2_path, B2_CHECKPOINT))
    for name, config, checkpoint in jobs:
        run([sys.executable, SCRIPTS / "benchmark.py", "--repo", REPO, "--config", config,
             "--checkpoint", checkpoint, "--device", DEVICE,
             "--output", OUT / name / "benchmark.json"], cwd=REPO)


## 9. Bảng kết quả và output cần giữ

APsmall toàn bộ 10 class và APsmall riêng phương tiện được ghi riêng. Recall-heavy kèm
số GT; ô trống nghĩa là nhóm không có GT. Chỉ chọn model trên validation; test/clip riêng
là giai đoạn sau. Notebook không tự suy ra P2 có lợi trước khi có E0/E1 cùng ngân sách.

In [ ]:
import pandas as pd
from IPython.display import display, FileLink
rows = []
for folder in sorted(OUT.iterdir()):
    metrics_path = folder / "export/evaluation/metrics.json"
    if not metrics_path.exists():
        continue
    m = json.loads(metrics_path.read_text())
    heavy = next(g for g in m["groups"] if g["scope"] == "vehicles" and g["group"] == "heavy")
    small_heavy = next(g for g in m["groups"] if g["scope"] == "vehicles" and g["group"] == "small_heavy")
    b = json.loads((folder / "benchmark.json").read_text()) if (folder / "benchmark.json").exists() else {}
    rows.append({"Model": folder.name, "Mode": RUN_MODE, "AP": m["all_classes"]["AP"],
                 "APsmall_all": m["all_classes"]["APsmall"], "AP_vehicle": m["vehicles"]["AP"],
                 "APsmall_vehicle": m["vehicles"]["APsmall"], "Recall_heavy_vehicle": heavy["recall"],
                 "GT_heavy_vehicle": heavy["gt"], "Recall_small_heavy_vehicle": small_heavy["recall"],
                 "GT_small_heavy_vehicle": small_heavy["gt"],
                 "FPS": b.get("FPS"), "Latency_ms": b.get("Latency_ms"), "Parameters_M": b.get("Parameters_M")})
table = pd.DataFrame(rows)
table.to_csv(OUT / "results.csv", index=False)
display(table.round(4))

# Artifact tự chứa config, GT, mapping, scripts, log và checkpoint cho phiên kế tiếp.
for filename in ("instances_train.json", "instances_val.json", "dataset_stats.json",
                 "rtdetr_dataset.json", "label_map.json", "classes.txt"):
    (OUT / "data").mkdir(exist_ok=True)
    shutil.copy2(COCO_ROOT / filename, OUT / "data" / filename)
shutil.copytree(SCRIPTS, OUT / "scripts", dirs_exist_ok=True)
(OUT / "repo_patch.diff").write_text(subprocess.check_output(
    ["git", "-C", str(REPO.parent), "diff"], text=True), encoding="utf-8")
(OUT / "environment.json").write_text(json.dumps({
    "repo_commit": REPO_COMMIT, "torch": str(torch.__version__),
    "torchvision": str(torchvision.__version__), "gpu": torch.cuda.get_device_name(0),
    "mode": RUN_MODE, "seed": SEED, "batch_size": BATCH_SIZE,
    "epochs": base["epoches"], "train_amp": USE_AMP,
    "train_annotations_sha256": TRAIN_HASH, "val_annotations_sha256": GT_HASH,
}, indent=2), encoding="utf-8")
archive = shutil.make_archive(str(WORK / f"rtdetrv2_p2_{RUN_MODE}"), "zip", OUT)
print("Output folder:", OUT)
print("Giữ best.pth để đánh giá; last.pth + config/manifest/log/best để resume.")
display(FileLink(Path(archive).name))


**Chạy tiếp theo kế hoạch:** thử `proxy` 2 epoch cho E0/E1 để xác nhận vòng train thật,
sau đó train `full` với cùng batch/seed cho hai model. Dùng `results.csv`,
`per_class.csv`, `recall_groups.csv` và ảnh lỗi để kiểm tra tín hiệu APsmall/Recall-heavy.
Output Kaggle phải được lưu trước khi kết thúc phiên; ZIP chỉ được tạo khi chạy tới cell cuối,
còn `last.pth`/`best.pth` đã nằm trong output từng experiment sau mỗi epoch.

Nguồn kiến trúc: [RT-DETR chính thức](https://github.com/lyuwenyu/RT-DETR/tree/29320b6fd828f8e0987a71426cf2d961b09dfed7/rtdetrv2_pytorch).